# 电影圈数据

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
from networkx.algorithms import bipartite

In [2]:
import sys
sys.path.append("..")

# 数据处理

## 原始数据

In [3]:
df_raw = pd.read_csv("dwd_cast_works.csv")
df_raw.head()

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,k_genres,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,喜剧/动作,无评分,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,爱情,无评分,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓
4,1013885,阿美莉嘉·奥利沃,女,10727641,演员,是,21,10727641,碟中谍5：神秘国度,电影,...,动作/惊悚/冒险,有评分,7.8,292193,292193.0,1510.0,21455331,2027819,5,Turandot


## 数据筛选
过滤条件
1. 电影
2. 有评分
3. 地区含中国

In [4]:
df_movie = df_raw.loc[(df_raw['k_type'] == '电影')
                      & (df_raw['k_region'].str.contains('中国'))].copy()
# k_cast_id转为str
df_movie['k_cast_id'] = df_movie['k_cast_id'].apply(lambda x: str(x))
# 仅保留演员，导演数
df_movie = df_movie[df_movie['k_role'].isin(['演员', '导演'])]
df_movie = df_movie.reset_index(drop=True).copy(deep=True)
# 添加新的movie_id_m列
df_movie['movie_id_m'] = df_movie['k_movie_id'].apply(lambda x: 'm' + str(x))
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,无评分,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN,m21534411
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,无评分,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN,m21236655
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,无评分,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN,m21202445
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,无评分,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓,m21133377
4,1003424,蒋君超,男,10777680,演员,是,2,10777680,影坛风月,电影,...,无评分,NaN,暂无评分,NaN,NaN,21555409,2006897,6,NaN,m21555409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224720,1370230,张巨光,男,3542526,演员,否,999,3542526,保卫胜利果实,电影,...,无评分,NaN,暂无评分,NaN,NaN,7085101,2740509,604126,村民,m7085101
224721,1431712,邓祥,男,1466295,演员,否,999,1466295,倚天屠龙记（上集）,电影,...,无评分,NaN,暂无评分,NaN,NaN,2932639,2863473,604127,史镖头,m2932639
224722,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,有评分,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243
224723,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,有评分,6.4,1123,1123.0,85.0,4541081,55131591,604130,NaN,m4541081


## 影人职责合并

In [5]:
# 合并影人职责
df_cast = df_movie[['k_cast_id', 'cast_name', 'k_role']].drop_duplicates()
df_cast_agg = df_cast.groupby(['k_cast_id', 'cast_name'])['k_role'].apply(lambda x: '/'.join(sorted(x.unique()))).reset_index()
df_cast_agg

,k_cast_id,cast_name,k_role
0,2000437,李香凝,演员
1,2000457,吉娜·卡拉诺,演员
2,2000515,理查德·格里克,演员
3,2000941,司汗,演员
4,2001011,凯文·格劳特,导演
...,...,...,...
42248,75617721,吴铃山,演员
42249,75617969,特蕾沙,演员
42250,75618493,何宥辰,演员
42251,75622879,祖丽米热,演员


In [6]:
df_movie['cast_role_agg'] = df_movie['k_cast_id'].map(
    df_cast_agg.set_index('k_cast_id')['k_role']
)
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN,m21534411,演员
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN,m21236655,演员
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN,m21202445,演员
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓,m21133377,演员
4,1003424,蒋君超,男,10777680,演员,是,2,10777680,影坛风月,电影,...,NaN,暂无评分,NaN,NaN,21555409,2006897,6,NaN,m21555409,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224720,1370230,张巨光,男,3542526,演员,否,999,3542526,保卫胜利果实,电影,...,NaN,暂无评分,NaN,NaN,7085101,2740509,604126,村民,m7085101,演员
224721,1431712,邓祥,男,1466295,演员,否,999,1466295,倚天屠龙记（上集）,电影,...,NaN,暂无评分,NaN,NaN,2932639,2863473,604127,史镖头,m2932639,演员
224722,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243,演员
224723,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,604130,NaN,m4541081,导演


In [7]:
df_movie[df_movie['cast_name'] == '张艺谋']

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
2234,27260166,张艺谋,男,1294963,导演,是,1,1294963,一个都不能少,电影,...,7.7,120767,120767.0,964.0,2589975,54520381,4242,NaN,m2589975,导演/演员
4017,27260166,张艺谋,男,35294995,演员,是,40,35294995,我和我的父辈,电影,...,6.9,175739,175739.0,1101.0,70590039,54520381,8991,电视台台长,m70590039,导演/演员
7580,27260166,张艺谋,男,1294007,导演,是,1,1294007,我的父亲母亲,电影,...,8.2,116620,116620.0,978.0,2588063,54520381,18378,NaN,m2588063,导演/演员
12782,27260166,张艺谋,男,1308070,演员,是,1,1308070,老井,电影,...,8.0,16354,16354.0,362.0,2616189,54520381,32540,孙旺泉,m2616189,导演/演员
17992,27260166,张艺谋,男,26694491,导演,是,1,26694491,大红灯笼高高挂,电影,...,8.8,3372,3372.0,172.0,53389031,54520381,46511,NaN,m53389031,导演/演员
19694,27260166,张艺谋,男,1297879,演员,否,999,1297879,大阅兵,电影,...,6.8,1946,1946.0,115.0,2595807,54520381,51101,军官,m2595807,导演/演员
48988,27260166,张艺谋,男,35215390,导演,是,1,35215390,狙击手,电影,...,7.7,340264,340264.0,1619.0,70430829,54520381,130724,NaN,m70430829,导演/演员
48990,27260166,张艺谋,男,33447633,导演,是,1,33447633,坚如磐石,电影,...,6.0,344998,344998.0,1439.0,66895315,54520381,130736,NaN,m66895315,导演/演员
61266,27260166,张艺谋,男,1296436,演员,是,8,1296436,有话好好说,电影,...,8.3,170972,170972.0,1191.0,2592921,54520381,163429,收废品的,m2592921,导演/演员
61267,27260166,张艺谋,男,1296436,导演,是,1,1296436,有话好好说,电影,...,8.3,170972,170972.0,1191.0,2592921,54520381,163430,NaN,m2592921,导演/演员


## 有评分数据

In [8]:
df_movie_rated = df_movie[df_movie['is_rating'] == '有评分'].copy()
df_movie_rated = df_movie_rated.reset_index(drop=True).copy(deep=True)
df_movie_rated

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,77,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,92,NaN,m21162685,演员
4,1315281,卢庆辉,男,10490155,演员,是,1,10490155,衰鬼抓狂,电影,...,5.8,121,121.0,26.0,20980359,2630611,156,NaN,m20980359,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100882,27541325,张世,男,1298203,演员,是,2,1298203,国道封闭,电影,...,7.9,146,146.0,34.0,2596455,55082699,604105,NaN,m2596455,导演/演员
100883,27517568,林迪安,男,1300498,演员,否,999,1300498,百变星君,电影,...,7.7,246162,246162.0,1377.0,2601045,55035185,604118,穿斑马服君,m2601045,演员
100884,27541395,韩鹏翼,男,24695588,演员,是,2,24695588,逆袭,电影,...,5.1,674,674.0,59.0,49391225,55082839,604119,NaN,m49391225,演员
100885,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243,演员


## 保存为csv

In [9]:
df_movie.to_csv("movie_data.csv", index=False)
df_movie_rated.to_csv("movie_data_rated.csv", index=False)

# 数据统计

## 年度数据

In [ ]:
# 按年统计每年的电影数量
df_count_by_year = df_movie.groupby('k_movie_year')['movie_id_m'].nunique().reset_index()
df_count_by_year.columns = ['year', 'movie_count']
df_count_by_year = df_count_by_year.sort_values('year')
df_count_by_year

In [ ]:
# 绘图
plt.figure(figsize=(10, 6))
plt.plot(df_count_by_year['year'], df_count_by_year['movie_count'], marker='o')
plt.title('年度电影数量统计')
plt.xlabel('年份')
plt.ylabel('电影数量')
plt.xticks(df_count_by_year['year'], rotation=45)
# plt.tight_layout()
plt.show()